# Portfolio Risk Management System - Testing Notebook

This notebook lets you run through the entire system step-by-step:

1. ✅ Test Bloomberg connection
2. ✅ Load Excel positions
3. ✅ Fetch live prices
4. ✅ Calculate risk metrics
5. ✅ Test SVB stress scenario
6. ✅ Visualize results

Run cells one at a time to see exactly what's happening!

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Add Python modules to path
sys.path.insert(0, str(Path.cwd() / 'portfolio_risk_system' / 'python'))

# Standard imports
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Our modules
from config import (
    TRADER_FILES,
    DEFAULT_VOL_TARGET,
    DEFAULT_CORRELATION,
    USE_LIVE_CORRELATIONS,
    FETCH_LIVE_PRICES,
    CORRELATION_LOOKBACK_DAYS,
    get_data_provider
)
from xlwings_report import (
    read_positions_openpyxl,
    read_settings_openpyxl,
    get_open_positions,
    get_closed_positions
)
from risk_calcs import (
    calculate_position_vol,
    calculate_portfolio_vol_uniform,
    calculate_portfolio_vol,
    calculate_2sigma_drawdowns,
    calculate_pnl
)
from svb_stress_data import SVBStressScenario

print("✅ All imports successful!")

## 2. Check Configuration

In [ ]:
print("Configuration Settings:")
print("=" * 50)
print(f"Default Vol Target: ${DEFAULT_VOL_TARGET}mm")
print(f"Default Correlation: {DEFAULT_CORRELATION}")
print(f"Use Live Correlations: {USE_LIVE_CORRELATIONS}")
print(f"Fetch Live Prices: {FETCH_LIVE_PRICES}")
print(f"Correlation Lookback: {CORRELATION_LOOKBACK_DAYS} days")
print("\nTrader Files:")
for trader_num, filepath in TRADER_FILES.items():
    exists = "✅" if filepath.exists() else "❌"
    print(f"  {exists} Trader {trader_num}: {filepath}")

## 3. Test Bloomberg Connection

In [ ]:
print("Testing Bloomberg Connection...")
print("=" * 50)

try:
    from xbbg import blp
    
    # Test with a simple ticker
    test_ticker = 'USSW10 Curncy'
    print(f"Fetching price for {test_ticker}...")
    
    result = blp.bdp(tickers=test_ticker, flds='PX_LAST')
    
    if result is not None and not result.empty:
        price = result.loc[test_ticker, 'px_last']
        print(f"✅ Bloomberg connected!")
        print(f"   {test_ticker}: {price}")
    else:
        print("⚠️ Bloomberg returned empty result")
        
except ImportError:
    print("❌ xbbg not installed (pip install xbbg)")
except Exception as e:
    print(f"❌ Bloomberg connection error: {e}")
    print("   Make sure Bloomberg Terminal is running")

## 4. Load Excel Files

In [ ]:
print("Loading Excel Files...")
print("=" * 50)

all_positions = []

for trader_num, filepath in TRADER_FILES.items():
    if not filepath.exists():
        print(f"❌ Trader {trader_num}: File not found")
        continue
    
    try:
        df = read_positions_openpyxl(filepath)
        df['trader_num'] = trader_num
        df['trader_name'] = f"Trader {trader_num}"
        all_positions.append(df)
        print(f"✅ Trader {trader_num}: {len(df)} positions loaded")
    except Exception as e:
        print(f"❌ Trader {trader_num}: Error - {e}")

if all_positions:
    df_all = pd.concat(all_positions, ignore_index=True)
    print(f"\n✅ Total: {len(df_all)} positions from {len(all_positions)} traders")
    
    # Show summary
    print("\nStatus Distribution:")
    print(df_all['status'].value_counts())
else:
    print("❌ No positions loaded!")
    df_all = pd.DataFrame()

## 5. Preview Data

In [ ]:
# Show first few positions
display_cols = ['trader_name', 'status', 'headline', 'ticker', 'direction', 'bpv', 'entry_level', 'current_level']
display_cols = [c for c in display_cols if c in df_all.columns]

print("Sample Positions:")
print("=" * 50)
df_all[display_cols].head(10)

## 6. Fetch Live Prices from Bloomberg

In [ ]:
if FETCH_LIVE_PRICES and not df_all.empty:
    print("Fetching Live Prices from Bloomberg...")
    print("=" * 50)
    
    tickers = df_all['ticker'].dropna().unique().tolist()
    print(f"Tickers to fetch: {len(tickers)}")
    
    try:
        from xbbg import blp
        result = blp.bdp(tickers=tickers, flds='PX_LAST')
        
        live_prices = {}
        if result is not None and not result.empty:
            for ticker in tickers:
                try:
                    if ticker in result.index:
                        price = result.loc[ticker, 'px_last']
                        if pd.notna(price):
                            live_prices[ticker] = float(price)
                except Exception as e:
                    print(f"⚠️ {ticker}: {e}")
        
        print(f"\n✅ Fetched {len(live_prices)} prices")
        
        # Update current_level in dataframe
        for idx, row in df_all.iterrows():
            ticker = row.get('ticker')
            if ticker and ticker in live_prices:
                df_all.at[idx, 'current_level'] = live_prices[ticker]
        
        print("\nSample Prices:")
        for ticker, price in list(live_prices.items())[:5]:
            print(f"  {ticker}: {price}")
            
    except Exception as e:
        print(f"❌ Error fetching prices: {e}")
        live_prices = {}
else:
    print("⚠️ Live price fetching disabled in config")
    live_prices = {}

## 7. Fetch Correlation Matrix from Bloomberg

In [ ]:
if USE_LIVE_CORRELATIONS and not df_all.empty:
    print("Fetching Correlation Matrix from Bloomberg...")
    print("=" * 50)
    
    tickers = df_all['ticker'].dropna().unique().tolist()
    print(f"Tickers: {len(tickers)}")
    print(f"Lookback: {CORRELATION_LOOKBACK_DAYS} days")
    
    try:
        data_provider = get_data_provider()
        corr_matrix = data_provider.get_correlation_matrix(tickers, CORRELATION_LOOKBACK_DAYS)
        
        if corr_matrix is not None and not corr_matrix.empty:
            print(f"\n✅ Correlation matrix: {corr_matrix.shape}")
            print("\nSample Correlations (first 3x3):")
            display(corr_matrix.iloc[:3, :3])
        else:
            print("⚠️ No correlation matrix returned")
            corr_matrix = None
            
    except Exception as e:
        print(f"❌ Error fetching correlation matrix: {e}")
        corr_matrix = None
else:
    print("⚠️ Live correlation fetching disabled in config")
    corr_matrix = None

## 8. Calculate Risk Metrics

In [ ]:
print("Calculating Risk Metrics...")
print("=" * 50)

# Filter to open positions
open_pos = get_open_positions(df_all)
print(f"Open positions: {len(open_pos)}")

if not open_pos.empty:
    # Calculate position volatility (using default 50bp for demo)
    default_vol_bp = 50
    open_pos['annual_vol_bp'] = default_vol_bp
    open_pos['position_vol'] = open_pos.apply(
        lambda row: calculate_position_vol(row.get('bpv', 0), default_vol_bp)
        if not pd.isna(row.get('bpv'))
        else 0,
        axis=1
    )
    
    # Calculate P&L
    def calc_pnl(row):
        entry = row.get('entry_level')
        current = row.get('current_level')
        bpv = row.get('bpv')
        direction = row.get('direction', '')
        
        if pd.isna(entry) or pd.isna(current) or pd.isna(bpv):
            return 0
        
        try:
            return calculate_pnl(entry, current, bpv, direction, 'IR Swap')
        except:
            return 0
    
    open_pos['pnl'] = open_pos.apply(calc_pnl, axis=1)
    
    # Calculate portfolio vol
    vols = open_pos['position_vol'].dropna()
    if len(vols) > 0:
        if corr_matrix is not None and not corr_matrix.empty:
            portfolio_vol = calculate_portfolio_vol(
                open_pos,
                correlation=DEFAULT_CORRELATION,
                corr_matrix=corr_matrix
            )
            corr_method = "Live Market Data"
        else:
            portfolio_vol = calculate_portfolio_vol_uniform(vols.values, DEFAULT_CORRELATION)
            corr_method = f"Fixed ({DEFAULT_CORRELATION})"
    else:
        portfolio_vol = 0
        corr_method = "N/A"
    
    # Calculate drawdowns
    drawdowns = calculate_2sigma_drawdowns(portfolio_vol)
    
    # Calculate other metrics
    total_pnl = open_pos['pnl'].sum()
    vol_target = DEFAULT_VOL_TARGET * 1_000_000
    headroom = vol_target - portfolio_vol
    risk_util = (portfolio_vol / vol_target) * 100
    
    print("\n📊 Portfolio Metrics:")
    print(f"  Total Book Vol: ${portfolio_vol:,.0f}")
    print(f"  Vol Target: ${vol_target:,.0f}")
    print(f"  Headroom: ${headroom:,.0f}")
    print(f"  Risk Utilization: {risk_util:.1f}%")
    print(f"  Total P&L: ${total_pnl:,.0f}")
    print(f"  Correlation Method: {corr_method}")
    
    print("\n📉 2-Sigma Drawdowns:")
    print(f"  1-Day: ${drawdowns['1d']:,.0f}")
    print(f"  1-Week: ${drawdowns['1w']:,.0f}")
    print(f"  2-Week: ${drawdowns['2w']:,.0f}")
    print(f"  1-Month: ${drawdowns['1m']:,.0f}")
else:
    print("⚠️ No open positions to calculate metrics")

## 9. SVB Stress Scenario

In [ ]:
if not open_pos.empty:
    print("SVB/CS Stress Scenario (March 2023)")
    print("=" * 50)
    
    svb = SVBStressScenario()
    stress_summary = svb.get_portfolio_stress_summary(open_pos)
    
    print(f"\n💥 Stress Test Results:")
    print(f"  Portfolio Stress P&L: ${stress_summary['total_stress_pnl']:,.0f}")
    print(f"  Worst Position: ${stress_summary['worst_position']:,.0f} ({stress_summary['worst_position_ticker']})")
    print(f"  Best Position: ${stress_summary['best_position']:,.0f} ({stress_summary['best_position_ticker']})")
    print(f"  Positions Covered: {stress_summary['positions_with_data']}/{stress_summary['total_positions']}")
    
    # Detail table
    stress_df = svb.calculate_portfolio_stress_pnl(open_pos)
    stress_detail = stress_df[stress_df['has_data']][['ticker', 'direction', 'bpv', 'svb_move', 'stress_pnl']]
    
    print("\nStress P&L by Position:")
    display(stress_detail)
else:
    print("⚠️ No open positions for stress test")

## 10. Risk by Trader

In [ ]:
if not open_pos.empty:
    print("Risk by Trader")
    print("=" * 50)
    
    trader_risk = open_pos.groupby('trader_name').agg({
        'trade_id': 'count',
        'position_vol': 'sum',
        'pnl': 'sum'
    }).reset_index()
    
    trader_risk.columns = ['Trader', '# Positions', 'Vol ($)', 'P&L ($)']
    trader_risk['% of Book'] = (trader_risk['Vol ($)'] / portfolio_vol * 100)
    
    display(trader_risk)
else:
    print("⚠️ No open positions")

## 11. Visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

if not open_pos.empty and 'trader_risk' in locals():
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Vol by Trader
    ax = axes[0, 0]
    trader_risk.plot(kind='bar', x='Trader', y='Vol ($)', ax=ax, legend=False, color='steelblue')
    ax.set_title('Vol by Trader')
    ax.set_ylabel('Vol ($)')
    ax.set_xlabel('')
    
    # P&L by Trader
    ax = axes[0, 1]
    colors = ['green' if x > 0 else 'red' for x in trader_risk['P&L ($)']]
    trader_risk.plot(kind='bar', x='Trader', y='P&L ($)', ax=ax, legend=False, color=colors)
    ax.set_title('P&L by Trader')
    ax.set_ylabel('P&L ($)')
    ax.set_xlabel('')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    
    # Drawdowns
    ax = axes[1, 0]
    dd_labels = ['1-Day', '1-Week', '2-Week', '1-Month']
    dd_values = [drawdowns['1d'], drawdowns['1w'], drawdowns['2w'], drawdowns['1m']]
    ax.bar(dd_labels, dd_values, color='coral')
    ax.set_title('2-Sigma Drawdowns')
    ax.set_ylabel('Drawdown ($)')
    
    # Risk Utilization
    ax = axes[1, 1]
    utilization = [risk_util, 100 - risk_util]
    labels = [f'Used\n{risk_util:.1f}%', f'Headroom\n{100-risk_util:.1f}%']
    colors_pie = ['steelblue', 'lightgray']
    ax.pie(utilization, labels=labels, colors=colors_pie, autopct='', startangle=90)
    ax.set_title('Risk Utilization')
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No data to visualize")

## 12. Quick Test - Individual Functions

In [ ]:
# Test individual risk calculations
print("Testing Individual Functions")
print("=" * 50)

# Position vol
bpv = 100000  # $100k per bp
vol_bp = 50   # 50bp annual vol
pos_vol = calculate_position_vol(bpv, vol_bp)
print(f"\nPosition Vol:")
print(f"  BPV: ${bpv:,}")
print(f"  Annual Vol: {vol_bp}bp")
print(f"  Position Vol: ${pos_vol:,.0f}")

# Portfolio vol (uniform correlation)
position_vols = np.array([5000000, 3000000, 4000000])  # $5mm, $3mm, $4mm
correlation = 0.3
port_vol = calculate_portfolio_vol_uniform(position_vols, correlation)
print(f"\nPortfolio Vol:")
print(f"  Position Vols: ${position_vols}")
print(f"  Correlation: {correlation}")
print(f"  Portfolio Vol: ${port_vol:,.0f}")

# Drawdowns
dd = calculate_2sigma_drawdowns(port_vol)
print(f"\n2-Sigma Drawdowns:")
for horizon, value in dd.items():
    print(f"  {horizon}: ${value:,.0f}")

# P&L calculation
entry = 3.5  # 3.5%
current = 3.8  # 3.8%
bpv = 100000
direction = "Pay"
pnl = calculate_pnl(entry, current, bpv, direction, 'IR Swap')
print(f"\nP&L Calculation:")
print(f"  Entry: {entry}%")
print(f"  Current: {current}%")
print(f"  BPV: ${bpv:,}")
print(f"  Direction: {direction}")
print(f"  P&L: ${pnl:,.0f}")

## Summary

You now have a complete walkthrough of the system!

### What This Notebook Does:

1. ✅ Tests Bloomberg connection
2. ✅ Loads all Excel files
3. ✅ Fetches live prices
4. ✅ Fetches correlation matrix
5. ✅ Calculates risk metrics
6. ✅ Runs SVB stress scenario
7. ✅ Creates visualizations
8. ✅ Tests individual functions

### Use Cases:

- **Development**: Test changes cell-by-cell
- **Debugging**: See exactly where things break
- **Learning**: Understand how calculations work
- **Testing**: Verify Bloomberg connections
- **Analysis**: Run custom queries on data

### Next Steps:

- Modify cells to test different scenarios
- Add your own analysis
- Export results to Excel/CSV
- Use as template for reports